In [ ]:
import os




VIMEO_DIR             = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet/sequences"
WORKING_DIR           = "/kaggle/working"

TRAIN_CSV_PATH        = os.path.join(WORKING_DIR, "micro_vfi_v6_TRAIN_HARD.csv")
TEST_CSV_PATH         = os.path.join(WORKING_DIR, "micro_vfi_v6_TEST_REALISTIC.csv")

MODEL_WEIGHTS_PATH    = os.path.join(WORKING_DIR, "best_micro_vfi_pro_v6.pth")
TRAIN_HISTORY_PATH    = os.path.join(WORKING_DIR, "training_history_pro_v6.csv")
TEST_REPORT_DIR       = os.path.join(WORKING_DIR, "micro_vfi_v6_final_reports")
ZIP_OUTPUT_NAME       = os.path.join(WORKING_DIR, "micro_vfi_v6_results")


TARGET_TRAIN_SAMPLES  = 15000  
TARGET_TEST_SAMPLES   = 2000   
MIN_CONTOUR_AREA      = 25     

LITERATUR_KOLAY_MAX_PX = 3.0
LITERATUR_ZOR_MIN_PX   = 8.0

TRAIN_GLOBAL_MAX      = 0.20   
TRAIN_PATCH_MIN_RATIO = 0.20   
TRAIN_PATCH_MAX_RATIO = 0.70   
TRAIN_DISP_MIN        = 4.0    
TRAIN_DISP_MAX        = 30.0  

TEST_GLOBAL_MAX       = 0.30   
TEST_PATCH_MIN_RATIO  = 0.05   
TEST_PATCH_MAX_RATIO  = 0.85   
TEST_DISP_MIN         = 0.5    
TEST_DISP_MAX         = 25.0


MAX_EPOCHS            = 200
BATCH_SIZE            = 1      
ACCUMULATION_STEPS    = 16     
INITIAL_LR            = 0.0001 

OSCILLATION_WINDOW    = 10      
OSCILLATION_TOLERANCE = 0.0008  
SCHEDULER_PATIENCE    = 3      
SCHEDULER_FACTOR      = 0.5    



NUM_TEST_SAMPLES      = 500    
STRIDE_CHECK          = 4      
BLUR_KERNEL_SIZE      = (21, 21) 

PATCH_AREA_BINS       = [0, 5000, 20000, float('inf')]
PATCH_AREA_LABELS     = ['1_Kucuk_Yama', '2_Orta_Yama', '3_Buyuk_Yama']
DISPLACEMENT_BINS     = [0.0, LITERATUR_KOLAY_MAX_PX, LITERATUR_ZOR_MIN_PX, float('inf')]
DISPLACEMENT_LABELS   = ['1_Kolay_Mikro_Hareket_0_3px', '2_Orta_Hareket_3_8px', '3_Zor_Makro_Hareket_8px_Ustu']

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(TEST_REPORT_DIR, exist_ok=True)


In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def fast_folder_scan(base_path):
    return [f2.path for f1 in os.scandir(base_path) if f1.is_dir() for f2 in os.scandir(f1.path) if f2.is_dir()]

def analyze_video_metrics(folder_path):
    try:
        p1, p3 = os.path.join(folder_path, "im1.png"), os.path.join(folder_path, "im3.png")
        img1, img3 = cv2.imread(p1, 0), cv2.imread(p3, 0)
        if img1 is None or img3 is None: return None
        h_orig, w_orig = img1.shape
        
        diff = cv2.absdiff(img1, img3)
        _, thresh = cv2.threshold(diff, 15, 255, cv2.THRESH_BINARY)
        global_ratio = np.count_nonzero(thresh) / (h_orig * w_orig)
        
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > MIN_CONTOUR_AREA] 
        if not valid_contours: return None
        
        x, y, w, h = cv2.boundingRect(np.concatenate(valid_contours))
        nw, nh = ((w + STRIDE_CHECK - 1) // STRIDE_CHECK) * STRIDE_CHECK, ((h + STRIDE_CHECK - 1) // STRIDE_CHECK) * STRIDE_CHECK
        nx, ny = max(0, x - (nw - w) // 2), max(0, y - (nh - h) // 2)
        if nx + nw > w_orig: nx = w_orig - nw
        if ny + nh > h_orig: ny = h_orig - nh
        
        patch_thresh = thresh[ny:ny+nh, nx:nx+nw]
        patch_ratio = np.count_nonzero(patch_thresh) / (nw * nh)
        
        patch1, patch3 = img1[ny:ny+nh, nx:nx+nw], img3[ny:ny+nh, nx:nx+nw]
        flow = cv2.calcOpticalFlowFarneback(patch1, patch3, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        valid_mag = mag[patch_thresh > 0]
        if len(valid_mag) == 0: return None
        avg_displacement = np.mean(valid_mag)
        
        v_id = folder_path.replace("\\", "/").split("/")[-2] + "_" + folder_path.replace("\\", "/").split("/")[-1]
        
        return {
            "Video_ID": v_id, "Folder_Path": folder_path,
            "Crop_Coords": f"{int(nx)},{int(ny)},{int(nx+nw)},{int(ny+nh)}",
            "Patch_Area": int(nw * nh), "Global_Ratio": round(global_ratio, 4),
            "Patch_Ratio": round(patch_ratio, 4), "Displacement_px": round(avg_displacement, 4)
        }
    except Exception: return None

if __name__ == '__main__':
    all_folders = fast_folder_scan(VIMEO_DIR)
    if all_folders:
        train_dataset, test_dataset, used_video_ids = [], [], set()
        pbar = tqdm(total=TARGET_TRAIN_SAMPLES + TARGET_TEST_SAMPLES, desc="🔍 V6 Çift Veri Seti Üretimi")

        with ThreadPoolExecutor(max_workers=4) as executor:
            for res in as_completed({executor.submit(analyze_video_metrics, f): f for f in all_folders}):
                res = res.result()
                if not res or res['Video_ID'] in used_video_ids: continue
                
                v_id, d_px, p_rat, g_rat = res['Video_ID'], res['Displacement_px'], res['Patch_Ratio'], res['Global_Ratio']

                if len(train_dataset) < TARGET_TRAIN_SAMPLES and (g_rat < TRAIN_GLOBAL_MAX and TRAIN_PATCH_MIN_RATIO <= p_rat <= TRAIN_PATCH_MAX_RATIO and TRAIN_DISP_MIN <= d_px <= TRAIN_DISP_MAX):
                    train_dataset.append(res); used_video_ids.add(v_id); pbar.update(1)
                
                elif len(test_dataset) < TARGET_TEST_SAMPLES and (g_rat < TEST_GLOBAL_MAX and TEST_PATCH_MIN_RATIO <= p_rat <= TEST_PATCH_MAX_RATIO and TEST_DISP_MIN <= d_px <= TEST_DISP_MAX):
                    test_dataset.append(res); used_video_ids.add(v_id); pbar.update(1)

                if len(train_dataset) >= TARGET_TRAIN_SAMPLES and len(test_dataset) >= TARGET_TEST_SAMPLES:
                    print("\n🎯 Veri setleri tamamlandı!"); break

        pbar.close()
        pd.DataFrame(train_dataset).to_csv(TRAIN_CSV_PATH, index=False)
        pd.DataFrame(test_dataset).to_csv(TEST_CSV_PATH, index=False)
        print(f"🚀 İŞLEM TAMAM! Eğitim (Zor): {TRAIN_CSV_PATH} | Test (Gerçekçi): {TEST_CSV_PATH}")

In [ ]:
import torch, cv2, os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MicroVFIDataset(Dataset):
    def __init__(self, csv_file): self.data = pd.read_csv(csv_file)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        x1, y1, x2, y2 = [int(c) for c in row['Crop_Coords'].split(',')]
        w, h = x2 - x1, y2 - y1
        nw, nh = ((w + STRIDE_CHECK - 1) // STRIDE_CHECK) * STRIDE_CHECK, ((h + STRIDE_CHECK - 1) // STRIDE_CHECK) * STRIDE_CHECK
        nx1, ny1 = max(0, x1 - (nw - w) // 2), max(0, y1 - (nh - h) // 2)
        nx2, ny2 = nx1 + nw, ny1 + nh
        
        im1 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im1.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
        im2 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im2.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
        im3 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im3.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
        
        return torch.from_numpy(im1).permute(2,0,1).float() / 255.0, torch.from_numpy(im2).permute(2,0,1).float() / 255.0, torch.from_numpy(im3).permute(2,0,1).float() / 255.0

class MicroVFINetPro(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.ReLU(True))
        self.drop = nn.Dropout2d(p=0.2) 
        self.enc3 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1, stride=2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(64, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(True))
        self.final = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(True), nn.Conv2d(16, 3, 3, padding=1), nn.Sigmoid())

    def get_coord_grids(self, x):
        b, c, h, w = x.size()
        y_grid = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(b, 1, h, w)
        x_grid = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(b, 1, h, w)
        return torch.cat([x_grid, y_grid], dim=1)

    def forward(self, im1, im3):
        x = torch.cat([im1, im3], dim=1)
        e1 = self.enc1(torch.cat([x, self.get_coord_grids(x)], dim=1))
        e2 = self.drop(self.enc2(e1))
        e3 = self.enc3(e2)
        d2 = self.dec2(e3)
        d1 = self.dec1(torch.cat([d2, e2], dim=1))
        return self.final(torch.cat([d1, e1], dim=1))

class SobelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        k_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        k_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.w_x = nn.Parameter(k_x, requires_grad=False).to(device)
        self.w_y = nn.Parameter(k_y, requires_grad=False).to(device)
        self.crit = nn.L1Loss()
    def forward(self, p, t):
        p_pad, t_pad = nn.functional.pad(p, (1, 1, 1, 1), mode='replicate'), nn.functional.pad(t, (1, 1, 1, 1), mode='replicate')
        return self.crit(nn.functional.conv2d(p_pad, self.w_x, groups=3), nn.functional.conv2d(t_pad, self.w_x, groups=3)) + \
               self.crit(nn.functional.conv2d(p_pad, self.w_y, groups=3), nn.functional.conv2d(t_pad, self.w_y, groups=3))

class OscillationDetector:
    def __init__(self, w_size, tol): self.w_size, self.tol, self.hist = w_size, tol, []
    def step(self, loss):
        self.hist.append(loss)
        if len(self.hist) > self.w_size: self.hist.pop(0)
        if len(self.hist) == self.w_size and (max(self.hist) - min(self.hist)) < self.tol:
            print("\n🛑 SALINIM (OSCILLATION) TETİKLENDİ! Model doygunlukta."); return True
        return False

if __name__ == '__main__':
    try:
        print("📦 ZORLU EĞİTİM SETİ (HARD) Yükleniyor...")
        dataset = MicroVFIDataset(TRAIN_CSV_PATH)
        tr_size = int(0.9 * len(dataset))
        tr_ds, val_ds = torch.utils.data.random_split(dataset, [tr_size, len(dataset) - tr_size])
        
        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        
        model = MicroVFINetPro().to(device)
        opt = optim.Adam(model.parameters(), lr=INITIAL_LR)
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE)        
        
        l1_crit, sobel_crit = nn.L1Loss(), SobelLoss()
        osc_det = OscillationDetector(OSCILLATION_WINDOW, OSCILLATION_TOLERANCE)
        best_loss, history = float('inf'), []
        
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train(); tr_loss = 0.0; opt.zero_grad()
            pbar = tqdm(tr_loader, desc=f"Ep {epoch}/{MAX_EPOCHS}", leave=False)
            for i, (im1, im2, im3) in enumerate(pbar):
                im1, im2, im3 = im1.to(device), im2.to(device), im3.to(device)
                pred = model(im1, im3)
                loss = l1_crit(pred, im2) + 0.1 * sobel_crit(pred, im2)
                loss.backward(); tr_loss += loss.item()
                
                if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(tr_loader):
                    opt.step(); opt.zero_grad()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})
                
            model.eval(); val_loss = 0.0
            with torch.no_grad():
                for im1, im2, im3 in val_loader:
                    im1, im2, im3 = im1.to(device), im2.to(device), im3.to(device)
                    val_loss += (l1_crit(model(im1, im3), im2) + 0.1 * sobel_crit(model(im1, im3), im2)).item()
                    
            avg_tr, avg_val, lr = tr_loss/len(tr_loader), val_loss/len(val_loader), opt.param_groups[0]['lr']
            print(f"📈 Ep {epoch:03d} | Train: {avg_tr:.5f} | Val: {avg_val:.5f} | LR: {lr:.6f}")
            
            history.append({'epoch': epoch, 'train_loss': avg_tr, 'val_loss': avg_val, 'lr': lr})
            pd.DataFrame(history).to_csv(TRAIN_HISTORY_PATH, index=False)
            
            if avg_val < best_loss:
                best_loss = avg_val; torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
                print(f"   🏆 En İyi Model Kaydedildi! (Loss: {best_loss:.5f})")
                
            sched.step(avg_val)
            if osc_det.step(avg_val): break

        print("🎉 EĞİTİM BİTTİ!")
    except Exception as e: print(f"\n❌ EĞİTİM HATASI (ZIRH DEVREDE): {e}")

In [4]:
import torch, cv2, os, shutil, time
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim

# ==============================================================================
# 🚀 MICRO-VFI PRO V6: GERÇEKÇİ TEST VE 5'Lİ KADEME ANALİZİ (HÜCRE 4)
# ==============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def calc_psnr(i1, i2): mse = np.mean((i1.astype(float) - i2.astype(float))**2); return 100.0 if mse==0 else 20 * np.log10(255.0 / np.sqrt(mse))
def calc_ssim(i1, i2): return ssim(i1, i2, data_range=255, channel_axis=2)

class MicroVFINetPro(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.ReLU(True))
        self.drop = nn.Dropout2d(p=0.2) 
        self.enc3 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1, stride=2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(64, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(True))
        self.final = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(True), nn.Conv2d(16, 3, 3, padding=1), nn.Sigmoid())
    def get_coord_grids(self, x):
        b, c, h, w = x.size()
        y_grid = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(b, 1, h, w)
        x_grid = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(b, 1, h, w)
        return torch.cat([x_grid, y_grid], dim=1)
    def forward(self, im1, im3):
        x = torch.cat([im1, im3], dim=1)
        e1 = self.enc1(torch.cat([x, self.get_coord_grids(x)], dim=1))
        e2 = self.drop(self.enc2(e1))
        e3 = self.enc3(e2)
        d2 = self.dec2(e3)
        d1 = self.dec1(torch.cat([d2, e2], dim=1))
        return self.final(torch.cat([d1, e1], dim=1))

def apply_blending(bg_img, patch_img, x1, y1, x2, y2):
    h, w = y2 - y1, x2 - x1
    mask = np.zeros((h, w), dtype=np.float32)
    pad_y, pad_x = max(1, int(h * 0.1)), max(1, int(w * 0.1))
    mask[pad_y:-pad_y, pad_x:-pad_x] = 1.0
    mask = np.stack([cv2.GaussianBlur(mask, BLUR_KERNEL_SIZE, 11)]*3, axis=-1)
    bg_crop = bg_img[y1:y2, x1:x2].astype(np.float32)
    res = bg_img.copy()
    res[y1:y2, x1:x2] = ((patch_img.astype(np.float32) * mask) + (bg_crop * (1.0 - mask))).astype(np.uint8)
    return res

def save_comprehensive_report(row, model, save_path, label_tag):
    folder, v_id, disp, psnr_val = row['Folder_Path'], row['Video_ID'], row.get('Displacement_px', 0), row.get('PSNR', 0)
    im1 = cv2.cvtColor(cv2.imread(os.path.join(folder, "im1.png")), cv2.COLOR_BGR2RGB)
    im2 = cv2.cvtColor(cv2.imread(os.path.join(folder, "im2.png")), cv2.COLOR_BGR2RGB)
    im3 = cv2.cvtColor(cv2.imread(os.path.join(folder, "im3.png")), cv2.COLOR_BGR2RGB)
    
    _, mask = cv2.threshold(cv2.absdiff(cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY), cv2.cvtColor(im3, cv2.COLOR_RGB2GRAY)), 15, 255, cv2.THRESH_BINARY)
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
    
    nx1, ny1, nx2, ny2 = [int(c) for c in row['Crop_Coords'].split(',')]
    cv2.rectangle(mask_rgb, (nx1, ny1), (nx2, ny2), (0, 255, 0), 2) 

    t1 = torch.from_numpy(im1[ny1:ny2, nx1:nx2]).permute(2,0,1).float().unsqueeze(0).to(device) / 255.0
    t3 = torch.from_numpy(im3[ny1:ny2, nx1:nx2]).permute(2,0,1).float().unsqueeze(0).to(device) / 255.0
    
    with torch.no_grad(): pred_patch = (model(t1, t3).squeeze(0).permute(1,2,0).cpu().numpy() * 255.0).astype(np.uint8)

    raw_pasted = im1.copy(); raw_pasted[ny1:ny2, nx1:nx2] = pred_patch 
    final_blended = apply_blending(im1, pred_patch, nx1, ny1, nx2, ny2)

    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    fig.suptitle(f"Sınıf: {label_tag} | Kayma: {disp:.1f}px | PSNR: {psnr_val:.2f}dB", fontsize=15, fontweight='bold', color='navy')
    axes[0,0].imshow(im1); axes[0,0].set_title("Frame 1")
    axes[0,1].imshow(im2); axes[0,1].set_title("Target Frame 2 (GT)")
    axes[0,2].imshow(im3); axes[0,2].set_title("Frame 3")
    axes[1,0].imshow(mask_rgb); axes[1,0].set_title("Maske & Kırpma Alanı")
    axes[1,1].imshow(raw_pasted); axes[1,1].set_title("Çiğ Yama Çıktısı")
    axes[1,2].imshow(final_blended); axes[1,2].set_title("Seamless Blending")
    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()

if __name__ == '__main__':
    try:
        DIR_CAT = os.path.join(TEST_REPORT_DIR, "1_Kategorize_Raporlar")
        DIR_5TIER = os.path.join(TEST_REPORT_DIR, "2_Hata_Derecelendirme_Raporlari")
        os.makedirs(DIR_CAT, exist_ok=True); os.makedirs(DIR_5TIER, exist_ok=True)
        
        model = MicroVFINetPro().to(device)
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device, weights_only=True))
        model.eval()

        test_df = pd.read_csv(TEST_CSV_PATH).sample(n=min(NUM_TEST_SAMPLES, pd.read_csv(TEST_CSV_PATH).shape[0]), random_state=42).reset_index(drop=True)
        results = []

        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="🔬 Gerçekçi Test İşleniyor"):
            nx1, ny1, nx2, ny2 = [int(c) for c in row['Crop_Coords'].split(',')]
            i1 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im1.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
            i2 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im2.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
            i3 = cv2.cvtColor(cv2.imread(os.path.join(row['Folder_Path'], "im3.png")), cv2.COLOR_BGR2RGB)[ny1:ny2, nx1:nx2]
            
            t1, t2, t3 = [torch.from_numpy(i).permute(2,0,1).float().unsqueeze(0).to(device) / 255.0 for i in [i1, i2, i3]]
            
            if device.type == 'cuda': torch.cuda.synchronize()
            st = time.time()
            with torch.no_grad(): pred = model(t1, t3)
            if device.type == 'cuda': torch.cuda.synchronize()
            
            pred_np = np.clip(pred.squeeze(0).permute(1,2,0).cpu().numpy() * 255.0, 0, 255).astype(np.uint8)
            results.append({**row.to_dict(), "PSNR": calc_psnr(i2, pred_np), "SSIM": calc_ssim(i2, pred_np), "Inference_ms": (time.time()-st)*1000.0})

        res_df = pd.DataFrame(results)
        
        # --- BÖLÜM A: LİTERATÜR BAZLI KATEGORİK ANALİZ ---
        res_df['Disp_Level'] = pd.cut(res_df['Displacement_px'], bins=DISPLACEMENT_BINS, labels=DISPLACEMENT_LABELS)
        res_df['Patch_Level'] = pd.cut(res_df['Patch_Area'], bins=PATCH_AREA_BINS, labels=PATCH_AREA_LABELS)
        res_df['Category'] = res_df['Patch_Level'].astype(str) + " ___ " + res_df['Disp_Level'].astype(str)
        
        print("\n" + "="*85)
        print(f"{'KATEGORİ (Literatür Bazlı)':<55} | Adet | PSNR(dB)")
        print("-" * 85)
        for cat, group in res_df.groupby('Category'):
            print(f"{cat:<55} | {len(group):<4} | {group['PSNR'].mean():.2f}")
            os.makedirs(os.path.join(DIR_CAT, cat.replace(" ___ ", "_")), exist_ok=True)
            save_comprehensive_report(group.sort_values('PSNR', ascending=False).iloc[0], model, os.path.join(DIR_CAT, cat.replace(" ___ ", "_"), "1_BEST.png"), cat)
        print("="*85)

        # --- BÖLÜM B: 5 KADEMELİ DERECELENDİRME ŞOVU ---
        print("\n🏆 5 Kademeli Bağımsız Hata Raporları Çiziliyor...")
        sorted_df = res_df.sort_values(by='PSNR', ascending=False).reset_index(drop=True)
        chunks = np.array_split(sorted_df, 5)
        tiers = ["1_Mukemmel_Sonuclar", "2_Iyi_Sonuclar", "3_Kabul_Edilebilir", "4_Zayif_Sonuclar", "5_En_Kotu_Hatalar"]
        
        for i, chunk in enumerate(chunks):
            tier_dir = os.path.join(DIR_5TIER, tiers[i])
            os.makedirs(tier_dir, exist_ok=True)
            for rank, (_, row) in enumerate(chunk.iloc[np.linspace(0, len(chunk)-1, 10, dtype=int)].iterrows()):
                save_comprehensive_report(row, model, os.path.join(tier_dir, f"Sira_{rank+1}_PSNR_{row['PSNR']:.1f}.png"), tiers[i])

        res_df.to_csv(os.path.join(TEST_REPORT_DIR, "tez_test_sonuclari.csv"), index=False)
        shutil.make_archive(ZIP_OUTPUT_NAME, 'zip', TEST_REPORT_DIR)
        print(f"\n✅ İŞLEM TAMAM! Dosyalar: {ZIP_OUTPUT_NAME}.zip")

    except Exception as e: print(f"\n❌ TEST HATASI (ZIRH DEVREDE): {e}")

🔬 Gerçekçi Test İşleniyor: 100%|██████████| 500/500 [00:44<00:00, 11.30it/s]



KATEGORİ (Literatür Bazlı)                              | Adet | PSNR(dB)
-------------------------------------------------------------------------------------
1_Kucuk_Yama ___ 2_Orta_Hareket_3_8px                   | 1    | 28.28
3_Buyuk_Yama ___ 1_Kolay_Mikro_Hareket_0_3px            | 25   | 29.91
3_Buyuk_Yama ___ 2_Orta_Hareket_3_8px                   | 370  | 28.24
3_Buyuk_Yama ___ 3_Zor_Makro_Hareket_8px_Ustu           | 104  | 26.27

🏆 5 Kademeli Bağımsız Hata Raporları Çiziliyor...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)



✅ İŞLEM TAMAM! Dosyalar: /kaggle/working/micro_vfi_v6_results.zip
